# U.S. Used Vehicle Market — Business Analysis

**Database:** `autos_db` | **View:** `vehicles_view`  
**Source:** ~400k Craigslist listings processed via AWS Glue → S3 Parquet → Athena  
**Pipeline:** `autos_etl.py` enriched each record with 5 KPIs: depreciation curve, regional demand, condition premium, mileage discount, deal quality score

This notebook executes the 8 business analysis queries from `queries/business_analysis.sql` against Amazon Athena, converts results to pandas DataFrames, and produces publication-ready visualizations.

## 0. Setup

In [ ]:
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import time
import io

# ── Athena config ──────────────────────────────────────────────────────────────
DATABASE        = 'autos_db'
OUTPUT_LOCATION = 's3://norman-autos-pipeline/athena-results/'
REGION          = 'us-east-1'

athena = boto3.client('athena', region_name=REGION)
s3     = boto3.client('s3',     region_name=REGION)

# ── Plot defaults ──────────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.facecolor': '#0f0f14',
    'axes.facecolor':   '#1a1a23',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#b0b0c0',
    'ytick.color':      '#b0b0c0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2a2a3a',
    'axes.edgecolor':   '#2a2a3a',
    'figure.titlesize': 14,
    'axes.titlesize':   12,
    'axes.labelsize':   10,
})
ACCENT = '#FF6600'
print('Setup complete')

## Athena Helper

In [ ]:
def run_query(sql: str, poll_interval: float = 1.5) -> pd.DataFrame:
    """Execute SQL on Athena and return a DataFrame. Raises on query failure."""
    resp = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': DATABASE},
        ResultConfiguration={'OutputLocation': OUTPUT_LOCATION},
    )
    qid = resp['QueryExecutionId']

    # Poll until terminal state
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)
        state  = status['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(poll_interval)

    if state != 'SUCCEEDED':
        reason = status['QueryExecution']['Status'].get('StateChangeReason', state)
        raise RuntimeError(f'Athena query {state}: {reason}')

    # Paginate results
    paginator = athena.get_paginator('get_query_results')
    rows, columns = [], None
    for page in paginator.paginate(QueryExecutionId=qid):
        result = page['ResultSet']
        if columns is None:
            columns = [c['Label'] for c in result['ResultSetMetadata']['ColumnInfo']]
        for row in result['Rows'][1 if columns and not rows else 0:]:
            rows.append([d.get('VarCharValue', None) for d in row['Data']])

    return pd.DataFrame(rows, columns=columns)


def to_numeric(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Cast columns to numeric, coercing errors to NaN."""
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df


print('Helper functions ready')

---
## Query 1 — Depreciation Curve by Manufacturer

**Business question:** Which manufacturers hold their value best as vehicles age?  
**Use case:** Helps buyers and dealers identify which brands command premium resale value across the vehicle lifecycle.

In [ ]:
SQL_Q1 = """
SELECT
  manufacturer,
  age_bracket,
  COUNT(*) AS listing_count,
  ROUND(AVG(price), 0) AS avg_price,
  ROUND(AVG(price) / NULLIF(AVG(avg_price_by_make_age), 0) * 100, 1) AS pct_of_benchmark,
  ROUND(AVG(price_per_mile), 4) AS avg_price_per_mile
FROM autos_db.vehicles_view
WHERE price > 0
  AND manufacturer IS NOT NULL
  AND avg_price_by_make_age > 0
GROUP BY manufacturer, age_bracket
HAVING COUNT(*) >= 10
ORDER BY manufacturer, age_bracket
"""

df_depr = run_query(SQL_Q1)
df_depr = to_numeric(df_depr, ['listing_count', 'avg_price', 'pct_of_benchmark', 'avg_price_per_mile'])
print(f'{len(df_depr):,} rows | {df_depr["manufacturer"].nunique()} manufacturers')
df_depr.head()

In [ ]:
# Top 10 manufacturers by total listing volume for the chart
top_makes = (
    df_depr.groupby('manufacturer')['listing_count'].sum()
    .nlargest(10).index.tolist()
)

age_order = ['0-3 years', '4-7 years', '8-12 years', '13+ years']
plot_data = (
    df_depr[df_depr['manufacturer'].isin(top_makes)]
    .assign(age_bracket=lambda x: pd.Categorical(x['age_bracket'], categories=age_order, ordered=True))
    .sort_values(['manufacturer', 'age_bracket'])
)

fig, ax = plt.subplots(figsize=(12, 6))
palette = sns.color_palette('tab10', n_colors=len(top_makes))
for i, make in enumerate(top_makes):
    d = plot_data[plot_data['manufacturer'] == make]
    ax.plot(d['age_bracket'], d['avg_price'], marker='o', label=make.title(),
            color=palette[i], linewidth=2)

ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Price Depreciation Curve — Top 10 Manufacturers by Volume', fontweight='bold', pad=12)
ax.set_xlabel('Vehicle Age Bracket')
ax.set_ylabel('Average Listing Price')
ax.legend(loc='upper right', fontsize=8, framealpha=0.3)
plt.tight_layout()
plt.savefig('../charts/q1_depreciation_curve.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## Query 2 — Sweet Spot Analysis: Best Value by Age and Mileage

**Business question:** What age/mileage combination delivers the best price-to-market ratio?  
**Use case:** Actionable inventory acquisition guidance — points to cohorts consistently priced below benchmark.

In [ ]:
SQL_Q2 = """
SELECT
  age_bracket,
  mileage_bucket,
  COUNT(*) AS listing_count,
  ROUND(AVG(price), 0) AS avg_price,
  ROUND(AVG(price) / NULLIF(AVG(avg_price_by_make_age), 0) * 100, 1) AS avg_pct_of_benchmark,
  SUM(CASE WHEN deal_quality = 'great_deal' THEN 1 ELSE 0 END) AS great_deal_count,
  ROUND(SUM(CASE WHEN deal_quality = 'great_deal' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS great_deal_pct
FROM autos_db.vehicles_view
WHERE price > 0
GROUP BY age_bracket, mileage_bucket
HAVING COUNT(*) >= 20
ORDER BY avg_pct_of_benchmark ASC
"""

df_sweet = run_query(SQL_Q2)
df_sweet = to_numeric(df_sweet, ['listing_count', 'avg_price', 'avg_pct_of_benchmark',
                                  'great_deal_count', 'great_deal_pct'])
print(f'{len(df_sweet):,} cohorts')
df_sweet.sort_values('avg_pct_of_benchmark').head(10)

In [ ]:
age_order    = ['0-3 years', '4-7 years', '8-12 years', '13+ years']
mile_order   = ['0-30k', '30-60k', '60-100k', '100-150k', '150k+']

pivot = (
    df_sweet
    .assign(
        age_bracket  = lambda x: pd.Categorical(x['age_bracket'],  categories=age_order,  ordered=True),
        mileage_bucket = lambda x: pd.Categorical(x['mileage_bucket'], categories=mile_order, ordered=True),
    )
    .pivot_table(index='age_bracket', columns='mileage_bucket',
                 values='avg_pct_of_benchmark', aggfunc='mean')
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    pivot, ax=ax, annot=True, fmt='.1f',
    cmap='RdYlGn_r',        # green = below benchmark (better deal)
    linewidths=0.5, linecolor='#0a0a0f',
    cbar_kws={'label': '% of benchmark price'},
    vmin=70, vmax=120,
)
ax.set_title('Sweet Spot Heatmap — Avg Price as % of Benchmark\n(greener = cheaper vs. benchmark = better deal)',
             fontweight='bold', pad=12)
ax.set_xlabel('Mileage Bucket')
ax.set_ylabel('Age Bracket')
plt.tight_layout()
plt.savefig('../charts/q2_sweet_spot_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## Query 3 — Regional Arbitrage Opportunities

**Business question:** Where is the same manufacturer selling significantly cheaper than the national median?  
**Threshold:** States where avg price is >15% below national median for the same manufacturer.

In [ ]:
SQL_Q3 = """
WITH national_median AS (
  SELECT
    manufacturer,
    APPROX_PERCENTILE(price, 0.5) AS national_median_price,
    COUNT(*) AS national_count
  FROM autos_db.vehicles_view
  WHERE price > 0 AND manufacturer IS NOT NULL
  GROUP BY manufacturer
  HAVING COUNT(*) >= 50
),
state_stats AS (
  SELECT
    state,
    manufacturer,
    ROUND(AVG(price), 0) AS state_avg_price,
    COUNT(*) AS state_count
  FROM autos_db.vehicles_view
  WHERE price > 0 AND manufacturer IS NOT NULL
  GROUP BY state, manufacturer
  HAVING COUNT(*) >= 5
)
SELECT
  s.state,
  s.manufacturer,
  s.state_avg_price,
  ROUND(n.national_median_price, 0) AS national_median_price,
  ROUND((s.state_avg_price - n.national_median_price) / n.national_median_price * 100, 1) AS pct_vs_national,
  s.state_count
FROM state_stats s
JOIN national_median n ON s.manufacturer = n.manufacturer
WHERE s.state_avg_price < n.national_median_price * 0.85
ORDER BY pct_vs_national ASC
LIMIT 30
"""

df_arb = run_query(SQL_Q3)
df_arb = to_numeric(df_arb, ['state_avg_price', 'national_median_price', 'pct_vs_national', 'state_count'])
print(f'{len(df_arb)} arbitrage opportunities identified')
df_arb.head(10)

In [ ]:
df_arb_plot = df_arb.copy()
df_arb_plot['label'] = df_arb_plot['state'] + ' / ' + df_arb_plot['manufacturer'].str.title()

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(df_arb_plot['label'], df_arb_plot['pct_vs_national'],
               color=ACCENT, edgecolor='none', height=0.7)
ax.axvline(0, color='#606070', linewidth=1)
ax.axvline(-15, color='#CC0000', linewidth=1, linestyle='--', alpha=0.6, label='-15% threshold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Regional Arbitrage — Top 30 State/Make Combos Below National Median\n(more negative = deeper discount vs national median)',
             fontweight='bold', pad=12)
ax.set_xlabel('% vs National Median Price')
ax.legend(fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../charts/q3_regional_arbitrage.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## Query 4 — Condition Premium Analysis

**Business question:** How much extra does "excellent" condition actually cost vs "good" or "fair"?  
**Use case:** Helps buyers decide whether condition upgrades are worth the premium per manufacturer.

In [ ]:
SQL_Q4 = """
SELECT
  manufacturer,
  condition,
  COUNT(*) AS listing_count,
  ROUND(AVG(price), 0) AS avg_price,
  ROUND(AVG(price) / NULLIF(
    AVG(AVG(price)) OVER (PARTITION BY manufacturer), 0) * 100, 1
  ) AS pct_of_make_avg
FROM autos_db.vehicles_view
WHERE price > 0
  AND manufacturer IS NOT NULL
  AND condition IS NOT NULL
  AND condition != ''
GROUP BY manufacturer, condition
HAVING COUNT(*) >= 10
ORDER BY manufacturer, avg_price DESC
"""

df_cond = run_query(SQL_Q4)
df_cond = to_numeric(df_cond, ['listing_count', 'avg_price', 'pct_of_make_avg'])
print(f'{len(df_cond):,} rows | {df_cond["manufacturer"].nunique()} manufacturers')
df_cond.head()

In [ ]:
# Condition premium spread = max - min avg_price within each manufacturer
spread = (
    df_cond.groupby('manufacturer')['avg_price']
    .apply(lambda x: x.max() - x.min())
    .nlargest(12)
    .reset_index()
    .rename(columns={'avg_price': 'price_spread'})
)

top_cond_makes = spread['manufacturer'].tolist()
plot_cond = df_cond[df_cond['manufacturer'].isin(top_cond_makes)].copy()

cond_order = ['like new', 'excellent', 'good', 'fair', 'salvage']
existing_conds = [c for c in cond_order if c in plot_cond['condition'].unique()]

pivot_cond = (
    plot_cond
    .pivot_table(index='manufacturer', columns='condition', values='avg_price')
    .reindex(columns=[c for c in cond_order if c in plot_cond['condition'].unique()])
    .loc[top_cond_makes]
)

fig, ax = plt.subplots(figsize=(12, 7))
pivot_cond.plot(kind='bar', ax=ax, width=0.75,
                color=sns.color_palette('coolwarm_r', n_colors=len(existing_conds)))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Condition Premium — Top 12 Manufacturers by Condition Price Spread',
             fontweight='bold', pad=12)
ax.set_xlabel('Manufacturer')
ax.set_ylabel('Average Price')
ax.set_xticklabels([m.title() for m in pivot_cond.index], rotation=35, ha='right')
ax.legend(title='Condition', fontsize=9)
plt.tight_layout()
plt.savefig('../charts/q4_condition_premium.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## Query 5 — Market Concentration and Price Elasticity by State

**Business question:** Do states with more inventory have lower prices? Does supply actually depress price?  
**Key metric:** Inventory rank vs price rank divergence — buyer-friendly vs seller-friendly markets.

In [ ]:
SQL_Q5 = """
SELECT
  state,
  COUNT(*) AS total_listings,
  state_listing_count,
  ROUND(AVG(price), 0) AS avg_price,
  ROUND(APPROX_PERCENTILE(price, 0.5), 0) AS median_price,
  SUM(CASE WHEN deal_quality = 'great_deal' THEN 1 ELSE 0 END) AS great_deals,
  ROUND(SUM(CASE WHEN deal_quality = 'great_deal' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS great_deal_pct,
  RANK() OVER (ORDER BY COUNT(*) DESC) AS inventory_rank,
  RANK() OVER (ORDER BY AVG(price) ASC) AS price_rank
FROM autos_db.vehicles_view
WHERE price > 0 AND state IS NOT NULL
GROUP BY state, state_listing_count
HAVING COUNT(*) >= 50
ORDER BY total_listings DESC
"""

df_market = run_query(SQL_Q5)
df_market = to_numeric(df_market, ['total_listings', 'state_listing_count', 'avg_price',
                                    'median_price', 'great_deals', 'great_deal_pct',
                                    'inventory_rank', 'price_rank'])
print(f'{len(df_market)} states')
df_market.head(10)

In [ ]:
df_market['rank_divergence'] = df_market['inventory_rank'] - df_market['price_rank']

fig, ax = plt.subplots(figsize=(11, 8))
sc = ax.scatter(
    df_market['total_listings'], df_market['median_price'],
    c=df_market['rank_divergence'],
    cmap='coolwarm', s=80, alpha=0.85, edgecolors='none'
)

# Label top/bottom outliers
label_states = pd.concat([
    df_market.nlargest(5, 'rank_divergence'),
    df_market.nsmallest(5, 'rank_divergence'),
    df_market.nlargest(3, 'total_listings'),
]).drop_duplicates('state')

for _, row in label_states.iterrows():
    ax.annotate(row['state'], (row['total_listings'], row['median_price']),
                fontsize=7, ha='left', va='bottom',
                xytext=(4, 3), textcoords='offset points', color='#c0c0d0')

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('Rank Divergence\n(+high = more inventory, still expensive = demand-driven)', fontsize=8)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Market Concentration vs. Price Elasticity by State\n'
             'Warm = high inventory but still expensive (demand-driven) | '
             'Cool = more inventory = lower price (supply-sensitive)',
             fontweight='bold', pad=12)
ax.set_xlabel('Total Listings')
ax.set_ylabel('Median Price')
plt.tight_layout()
plt.savefig('../charts/q5_market_concentration.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## Query 6 — Deal Quality Distribution by Manufacturer and State

**Business question:** Which manufacturer/state combinations have the highest concentration of underpriced inventory?  
**Use case:** Sourcing intelligence for dealers — ranks combos by great deal density within each state.

In [ ]:
SQL_Q6 = """
SELECT
  manufacturer,
  state,
  COUNT(*) AS total_listings,
  SUM(CASE WHEN deal_quality = 'great_deal' THEN 1 ELSE 0 END) AS great_deals,
  SUM(CASE WHEN deal_quality = 'good_deal'  THEN 1 ELSE 0 END) AS good_deals,
  SUM(CASE WHEN deal_quality = 'fair'       THEN 1 ELSE 0 END) AS fair_deals,
  SUM(CASE WHEN deal_quality = 'overpriced' THEN 1 ELSE 0 END) AS overpriced,
  ROUND(SUM(CASE WHEN deal_quality IN ('great_deal','good_deal') THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_good_or_better,
  RANK() OVER (PARTITION BY state ORDER BY SUM(CASE WHEN deal_quality = 'great_deal' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) DESC) AS opportunity_rank
FROM autos_db.vehicles_view
WHERE price > 0
  AND manufacturer IS NOT NULL
  AND state IS NOT NULL
GROUP BY manufacturer, state
HAVING COUNT(*) >= 10
ORDER BY pct_good_or_better DESC
LIMIT 40
"""

df_deal = run_query(SQL_Q6)
df_deal = to_numeric(df_deal, ['total_listings', 'great_deals', 'good_deals',
                                'fair_deals', 'overpriced', 'pct_good_or_better', 'opportunity_rank'])
print(f'{len(df_deal)} combos')
df_deal.head(10)

In [ ]:
top40 = df_deal.head(40).copy()
top40['label'] = top40['manufacturer'].str.title() + ' / ' + top40['state']

stacked_cols  = ['great_deals', 'good_deals', 'fair_deals', 'overpriced']
stacked_colors = ['#2ecc71', '#a8e6a3', '#f0c040', '#e74c3c']
stacked_labels = ['Great Deal', 'Good Deal', 'Fair', 'Overpriced']

fig, ax = plt.subplots(figsize=(11, 12))
left = np.zeros(len(top40))
for col, color, label in zip(stacked_cols, stacked_colors, stacked_labels):
    ax.barh(top40['label'], top40[col], left=left, color=color, label=label, height=0.75)
    left += top40[col].values

ax.set_title('Deal Quality Distribution — Top 40 Manufacturer/State Combos\n(ranked by % good-or-better deals)',
             fontweight='bold', pad=12)
ax.set_xlabel('Listing Count')
ax.legend(loc='lower right', fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../charts/q6_deal_quality.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## Query 7 — Price Outlier Detection (Z-Score)

**Business question:** Which listings are priced anomalously vs their cohort?  
**Use case:** (1) Data quality flags. (2) Genuine bargain alerts — z < -2 within a statistically reliable cohort.

In [ ]:
SQL_Q7 = """
WITH cohort_stats AS (
  SELECT
    manufacturer,
    age_bracket,
    mileage_bucket,
    AVG(price)    AS cohort_avg,
    STDDEV(price) AS cohort_stddev,
    COUNT(*)      AS cohort_size
  FROM autos_db.vehicles_view
  WHERE price > 0 AND manufacturer IS NOT NULL
  GROUP BY manufacturer, age_bracket, mileage_bucket
  HAVING COUNT(*) >= 5
)
SELECT
  v.manufacturer,
  v.state,
  v.price,
  v.condition,
  v.vehicle_age,
  v.odometer,
  v.deal_quality,
  ROUND(c.cohort_avg,    0) AS cohort_avg_price,
  ROUND(c.cohort_stddev, 0) AS cohort_stddev,
  ROUND((v.price - c.cohort_avg) / NULLIF(c.cohort_stddev, 0), 2) AS z_score,
  CASE
    WHEN (v.price - c.cohort_avg) / NULLIF(c.cohort_stddev, 0) < -2 THEN 'significant_bargain'
    WHEN (v.price - c.cohort_avg) / NULLIF(c.cohort_stddev, 0) > 2  THEN 'significant_outlier'
    ELSE 'normal'
  END AS outlier_flag
FROM autos_db.vehicles_view v
JOIN cohort_stats c
  ON v.manufacturer = c.manufacturer
  AND v.age_bracket  = c.age_bracket
  AND v.mileage_bucket = c.mileage_bucket
WHERE price > 0
  AND ABS((v.price - c.cohort_avg) / NULLIF(c.cohort_stddev, 0)) > 2
ORDER BY z_score ASC
LIMIT 50
"""

df_outlier = run_query(SQL_Q7)
df_outlier = to_numeric(df_outlier, ['price', 'vehicle_age', 'odometer',
                                      'cohort_avg_price', 'cohort_stddev', 'z_score'])
print(f'{len(df_outlier)} outlier records returned')
df_outlier[['manufacturer', 'state', 'price', 'cohort_avg_price', 'z_score', 'outlier_flag']].head(10)

In [ ]:
bargains  = df_outlier[df_outlier['outlier_flag'] == 'significant_bargain']
high_out  = df_outlier[df_outlier['outlier_flag'] == 'significant_outlier']

fig, ax = plt.subplots(figsize=(10, 5))
ax.axvspan(-10, -2, alpha=0.12, color='#2ecc71', label='Bargain zone (z < -2)')
ax.axvspan(2,  10,  alpha=0.12, color='#e74c3c', label='Overpriced zone (z > 2)')
ax.axvline(-2, color='#2ecc71', linewidth=1, linestyle='--', alpha=0.5)
ax.axvline( 2, color='#e74c3c', linewidth=1, linestyle='--', alpha=0.5)
ax.scatter(bargains['z_score'],  np.random.RandomState(42).uniform(0.1, 0.9, len(bargains)),
           color='#2ecc71', s=60, alpha=0.8, label=f'Bargains ({len(bargains)})')
ax.scatter(high_out['z_score'], np.random.RandomState(43).uniform(0.1, 0.9, len(high_out)),
           color='#e74c3c', s=60, alpha=0.8, label=f'Outliers ({len(high_out)})')
ax.set_title('Price Outlier Distribution — Z-Scores Beyond ±2σ (cohort-relative)',
             fontweight='bold', pad=12)
ax.set_xlabel('Z-Score (standard deviations from cohort average)')
ax.set_yticks([])
ax.set_xlim(-8, 8)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../charts/q7_outlier_zscore.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## Query 8 — Duplicate Listing Detection

**Business question:** How much of the inventory is duplicate or re-listed? How much does this inflate apparent market size?  
**Definition:** Exact match on price + manufacturer + model + odometer + state.

In [ ]:
SQL_Q8 = """
SELECT
  price,
  manufacturer,
  model,
  odometer,
  state,
  COUNT(*) AS duplicate_count
FROM autos_db.vehicles_view
WHERE price > 0
  AND manufacturer IS NOT NULL
GROUP BY price, manufacturer, model, odometer, state
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 30
"""

df_dupes = run_query(SQL_Q8)
df_dupes = to_numeric(df_dupes, ['price', 'odometer', 'duplicate_count'])
print(f'Top {len(df_dupes)} duplicate groups')

total_dupe_listings   = df_dupes['duplicate_count'].sum()
unique_dupe_vehicles  = len(df_dupes)
inflated_listings     = total_dupe_listings - unique_dupe_vehicles
print(f'\nDuplicate analysis (top-30 groups):')
print(f'  Total listings in these groups : {total_dupe_listings:,}')
print(f'  Unique vehicles represented    : {unique_dupe_vehicles:,}')
print(f'  Inflated listing count         : {inflated_listings:,}')
df_dupes.head(10)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: top duplicated makes
make_dupes = (
    df_dupes.groupby('manufacturer')['duplicate_count'].sum()
    .sort_values(ascending=True)
    .tail(12)
)
ax1.barh(make_dupes.index.str.title(), make_dupes.values, color=ACCENT, height=0.7)
ax1.set_title('Duplicate Listings by Manufacturer\n(top 12 by duplicate count)', fontweight='bold')
ax1.set_xlabel('Duplicate Listing Count')

# Right: distribution of duplicate_count values
ax2.hist(df_dupes['duplicate_count'], bins=range(2, df_dupes['duplicate_count'].max() + 2),
         color='#4FC3F7', edgecolor='#0a0a0f', align='left')
ax2.set_title('Distribution of Re-listing Frequency\n(among groups with >1 listing)', fontweight='bold')
ax2.set_xlabel('Number of Duplicate Copies')
ax2.set_ylabel('Number of Vehicle Groups')

plt.suptitle('Duplicate / Re-listed Vehicle Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../charts/q8_duplicate_analysis.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## Key Findings Summary

In [ ]:
print('=== U.S. USED VEHICLE MARKET — KEY FINDINGS ===')
print()

# Q1: steepest depreciation make
depr_spread = (
    df_depr.groupby('manufacturer')['avg_price']
    .apply(lambda x: (x.max() - x.min()) / x.max() * 100 if x.max() > 0 else 0)
    .sort_values(ascending=False)
)
print(f'[Q1] Steepest depreciation: {depr_spread.idxmax().title()} '
      f'({depr_spread.max():.0f}% drop from new to 13+ years)')

# Q2: best sweet spot
best_cell = df_sweet.sort_values('avg_pct_of_benchmark').iloc[0]
print(f'[Q2] Best sweet spot: {best_cell["age_bracket"]} / {best_cell["mileage_bucket"]} '
      f'@ {best_cell["avg_pct_of_benchmark"]:.1f}% of benchmark '
      f'({best_cell["great_deal_pct"]:.1f}% great deals)')

# Q3: deepest arbitrage
if len(df_arb) > 0:
    best_arb = df_arb.sort_values('pct_vs_national').iloc[0]
    print(f'[Q3] Deepest arbitrage: {best_arb["manufacturer"].title()} in {best_arb["state"]} '
          f'({best_arb["pct_vs_national"]:.1f}% vs national median)')

# Q4: biggest condition premium make
best_cond_make = (
    df_cond.groupby('manufacturer')['avg_price']
    .apply(lambda x: x.max() - x.min())
    .idxmax()
)
best_cond_spread = (
    df_cond.groupby('manufacturer')['avg_price']
    .apply(lambda x: x.max() - x.min())
    .max()
)
print(f'[Q4] Largest condition premium: {best_cond_make.title()} '
      f'(${best_cond_spread:,.0f} spread between best and worst condition)')

# Q5: most buyer-friendly state (lowest price rank vs inventory)
buyer_friendly = df_market.sort_values('rank_divergence').iloc[0]
print(f'[Q5] Most buyer-friendly: {buyer_friendly["state"]} '
      f'(inventory rank {buyer_friendly["inventory_rank"]:.0f}, '
      f'price rank {buyer_friendly["price_rank"]:.0f})')

# Q6: highest great-deal concentration
best_deal = df_deal.sort_values('pct_good_or_better', ascending=False).iloc[0]
print(f'[Q6] Highest deal density: {best_deal["manufacturer"].title()} in {best_deal["state"]} '
      f'({best_deal["pct_good_or_better"]:.1f}% good-or-better deals)')

# Q7: bargain count
bargain_count = len(df_outlier[df_outlier['outlier_flag'] == 'significant_bargain'])
print(f'[Q7] Significant bargains identified: {bargain_count} listings with z < -2 in this sample')

# Q8: duplicate inflation
print(f'[Q8] Duplicate inflation: {inflated_listings:,} redundant listings '
      f'in the top-30 duplicate groups alone')